# 04 · Autoregressive Generation
### Practical LLM Engineering — Module 01: Fundamentals

> **Objectives**
> - How LLMs generate text one token at a time
> - The full generate loop: prefill → decode → stop
> - How the KV cache eliminates redundant computation
> - How to measure and profile generation throughput
> - Engineering tradeoffs: latency vs throughput vs memory

---


## 1. Overview

LLMs do not output full sentences. They output **one token at a time**, in a loop:

```
Prompt:  "The capital of France is"
Step 1:  "The capital of France is" → " Paris"
Step 2:  "The capital of France is Paris" → "."
Step 3:  "The capital of France is Paris." → "<eos>"
                                                ↑ stop
```

This is **autoregressive generation**: each new token is conditioned on all previous tokens, including the ones the model generated itself.

Formally, an LLM models the joint probability of a sequence as a product of conditionals:

$$
P(x_1, x_2, \ldots, x_T) = \prod_{t=1}^{T} P(x_t \mid x_1, \ldots, x_{t-1})
$$

At each step, the model produces a **logit vector** over the full vocabulary.
A **decoding strategy** turns those logits into the next token.

---

### Two phases of generation

| Phase | What happens | Compute pattern |
|---|---|---|
| **Prefill** | Process the entire prompt in one forward pass | High parallelism, fast |
| **Decode** | Generate one token per forward pass, autoregressively | Sequential, slower |

The decode phase is where the **KV cache** becomes critical.


## 2. Setup

In [ ]:
# Install dependencies (run once on Colab)
# !pip install torch transformers matplotlib numpy -q

import math
import time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Libraries ready  |  device={device}")


## 3. Background: The Autoregressive Objective

### 3.1 Next-token prediction

During training, the model learns to predict the next token at every position simultaneously (teacher forcing):

$$
\mathcal{L} = -\frac{1}{T} \sum_{t=1}^{T} \log P_\theta(x_t \mid x_1, \ldots, x_{t-1})
$$

At inference, we sample from this distribution one token at a time.

---

### 3.2 From logits to probabilities

The model's final linear layer produces **logits** $\mathbf{z} \in \mathbb{R}^{|\mathcal{V}|}$, one per vocabulary token.

These are converted to probabilities via **softmax with temperature** $\tau$:

$$
P(x_t = v) = \frac{\exp(z_v / \tau)}{\sum_{v'} \exp(z_{v'} / \tau)}
$$

Temperature $\tau$ controls the sharpness of the distribution:
- $\tau \to 0$: deterministic (greedy) — always picks the highest logit
- $\tau = 1$: standard sampling from the model distribution
- $\tau > 1$: flatter distribution — more random / creative

---

### 3.3 Stop conditions

Generation stops when any of these occur:
1. The model produces an **EOS** (end-of-sequence) token
2. A **maximum length** (`max_new_tokens`) is reached
3. A **stop string** is matched (e.g. `"\n\n"`, `"Human:"`)


## 4. The Generate Loop — From Scratch

In [ ]:
# ── Minimal autoregressive generate loop ─────────────────────────────
def generate(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 50,
    temperature: float = 1.0,
    greedy: bool = False,
    device: str = "cpu",
) -> str:
    """
    Bare-bones autoregressive generation loop.
    No KV cache — recomputes full attention at every step.
    """
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated  = input_ids.clone()

    print(f"Prompt tokens : {input_ids.shape[1]}")
    print(f"Generating up to {max_new_tokens} tokens...\n")

    step_times = []

    with torch.no_grad():
        for step in range(max_new_tokens):
            t0 = time.perf_counter()

            # Full forward pass over the ENTIRE sequence so far
            outputs = model(generated)
            logits  = outputs.logits[:, -1, :]   # (1, vocab_size) — last position only

            # Apply temperature
            if temperature != 1.0:
                logits = logits / temperature

            # Decode: greedy or sample
            if greedy:
                next_token = logits.argmax(dim=-1, keepdim=True)   # (1, 1)
            else:
                probs      = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)

            step_times.append(time.perf_counter() - t0)

            # Append to sequence
            generated = torch.cat([generated, next_token], dim=1)

            # Stop on EOS
            if next_token.item() == tokenizer.eos_token_id:
                print(f"[EOS at step {step+1}]")
                break

    output_text = tokenizer.decode(generated[0], skip_special_tokens=True)
    n_new       = generated.shape[1] - input_ids.shape[1]

    print(f"Generated {n_new} tokens")
    print(f"Avg step time : {np.mean(step_times)*1000:.1f} ms/token")
    print(f"Throughput    : {1/np.mean(step_times):.1f} tokens/sec")
    print()
    print("Output:")
    print(output_text)
    return output_text, step_times


In [ ]:
# ── Load a small model (GPT-2 runs on CPU) ───────────────────────────
print("Loading GPT-2...")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
model     = AutoModelForCausalLM.from_pretrained("gpt2").to(device)

print(f"Model parameters : {sum(p.numel() for p in model.parameters()):,}")
print(f"Vocabulary size  : {tokenizer.vocab_size:,}\n")

text, step_times_no_cache = generate(
    model, tokenizer,
    prompt="The transformer architecture was introduced in",
    max_new_tokens=30,
    greedy=True,
    device=device,
)


## 5. The KV Cache

### 5.1 The problem with naive generation

In the loop above, at step $t$ we feed the model the entire sequence $x_1, \ldots, x_t$.
The attention mechanism computes $\mathbf{K}$ and $\mathbf{V}$ for **all** $t$ tokens — even though $x_1, \ldots, x_{t-1}$ haven't changed since the last step.

This is pure **redundant computation**:

```
Step 1:  compute K,V for [x₁]
Step 2:  compute K,V for [x₁, x₂]   ← recomputes x₁
Step 3:  compute K,V for [x₁, x₂, x₃] ← recomputes x₁, x₂
...
Step T:  compute K,V for [x₁,...,xT]  ← recomputes everything
```

Total KV computations: $\mathcal{O}(T^2)$ — quadratic in generation length.

---

### 5.2 The solution: cache K and V

The **KV cache** stores the key and value tensors for all previous tokens.
At step $t$, we only need to:
1. Compute Q, K, V for the **new token** $x_t$
2. **Append** $K_t$, $V_t$ to the cache
3. Attend over the full cached $K_{1:t}$, $V_{1:t}$

```
Step 1:  compute K,V for [x₁]              cache = [K₁, V₁]
Step 2:  compute K,V for [x₂] only         cache = [K₁, K₂ | V₁, V₂]
Step 3:  compute K,V for [x₃] only         cache = [K₁, K₂, K₃ | ...]
```

Total KV computations: $\mathcal{O}(T)$ — **linear** in generation length.

---

### 5.3 KV cache memory cost

$$
\text{KV cache size} = 2 \times n_{\text{layers}} \times n_{\text{kv\_heads}} \times d_k \times T_{\text{max}} \times \text{bytes}
$$

For LLaMA-3-8B (float16):
- $n_{\text{layers}} = 32$, $n_{\text{kv\_heads}} = 8$, $d_k = 128$, $T = 8192$

$$
= 2 \times 32 \times 8 \times 128 \times 8192 \times 2 \approx \mathbf{1.07\text{ GB}}
$$


In [ ]:
# ── Visualise: naive vs KV cache compute ─────────────────────────────
gen_lengths = list(range(1, 51))

# Naive: at step t, compute KV for all t tokens → total = sum(1..T) = T(T+1)/2
naive_ops = [t * (t + 1) / 2 for t in gen_lengths]

# KV cache: at step t, compute KV for 1 token only → total = T
cache_ops = gen_lengths

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(gen_lengths, naive_ops, "o-", color="#e74c3c", label="Naive (no cache)", lw=2)
axes[0].plot(gen_lengths, cache_ops, "s-", color="#2ecc71", label="KV cache", lw=2)
axes[0].set_xlabel("Tokens generated")
axes[0].set_ylabel("Total KV compute (relative units)")
axes[0].set_title("KV compute: naive O(T²) vs cached O(T)")
axes[0].legend(); axes[0].grid(alpha=0.3)

speedup = [n / c for n, c in zip(naive_ops, cache_ops)]
axes[1].plot(gen_lengths, speedup, "^-", color="#9b59b6", lw=2)
axes[1].set_xlabel("Tokens generated")
axes[1].set_ylabel("Speedup (×)")
axes[1].set_title("KV cache speedup over naive generation")
axes[1].grid(alpha=0.3)
axes[1].fill_between(gen_lengths, 1, speedup, alpha=0.15, color="#9b59b6")

plt.tight_layout()
plt.show()

print(f"At 50 tokens: KV cache is {speedup[-1]:.0f}× fewer KV computations")


In [ ]:
# ── Generation WITH KV cache (HuggingFace built-in) ──────────────────
def generate_with_cache(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 50,
    greedy: bool = True,
    device: str = "cpu",
):
    """Generation using HuggingFace's built-in KV cache (use_cache=True)."""
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated  = input_ids.clone()
    past_key_values = None   # will hold the KV cache

    step_times = []

    with torch.no_grad():
        for step in range(max_new_tokens):
            t0 = time.perf_counter()

            outputs = model(
                generated if past_key_values is None else generated[:, -1:],
                past_key_values=past_key_values,
                use_cache=True,
            )
            logits          = outputs.logits[:, -1, :]
            past_key_values = outputs.past_key_values   # updated cache

            next_token = logits.argmax(dim=-1, keepdim=True) if greedy                          else torch.multinomial(F.softmax(logits, dim=-1), 1)

            step_times.append(time.perf_counter() - t0)
            generated = torch.cat([generated, next_token], dim=1)

            if next_token.item() == tokenizer.eos_token_id:
                break

    # Report cache size
    kv = past_key_values[0]  # (K, V) for layer 0
    k_shape = kv[0].shape
    print(f"KV cache — layer 0 key shape: {k_shape}")
    print(f"  = (batch={k_shape[0]}, heads={k_shape[1]}, "
          f"seq={k_shape[2]}, d_k={k_shape[3]})")

    return tokenizer.decode(generated[0], skip_special_tokens=True), step_times


prompt = "The transformer architecture was introduced in"
_, step_times_cache = generate_with_cache(
    model, tokenizer, prompt, max_new_tokens=30, greedy=True, device=device
)


In [ ]:
# ── Benchmark: no cache vs cache ─────────────────────────────────────
print("Timing comparison (30 tokens):")
print(f"  No KV cache  : {np.mean(step_times_no_cache)*1000:.1f} ms/token  "
      f"({1/np.mean(step_times_no_cache):.1f} tok/s)")
print(f"  With KV cache: {np.mean(step_times_cache)*1000:.1f} ms/token  "
      f"({1/np.mean(step_times_cache):.1f} tok/s)")
print(f"  Speedup      : {np.mean(step_times_no_cache)/np.mean(step_times_cache):.2f}×")
print()

# Plot step-by-step latency
fig, ax = plt.subplots(figsize=(10, 4))
steps = range(1, len(step_times_no_cache) + 1)
ax.plot(steps, [t * 1000 for t in step_times_no_cache],
        "o-", color="#e74c3c", label="No KV cache", lw=2, alpha=0.8)
ax.plot(steps[:len(step_times_cache)],
        [t * 1000 for t in step_times_cache],
        "s-", color="#2ecc71", label="With KV cache", lw=2, alpha=0.8)
ax.set_xlabel("Generation step")
ax.set_ylabel("Time per step (ms)")
ax.set_title("Per-step latency: KV cache eliminates growing cost")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 6. Prefill vs Decode Phase

Every generation request has two distinct phases with very different compute profiles:

```
┌─────────────────────────────────────────────────────────┐
│  PREFILL                                                  │
│  Input: full prompt (n_prompt tokens)                     │
│  Output: KV cache for all prompt tokens                   │
│  Compute: parallel — processes all tokens simultaneously  │
│  Duration: ~proportional to prompt length                 │
└─────────────────────────────────────────────────────────┘
                         ↓
┌─────────────────────────────────────────────────────────┐
│  DECODE                                                   │
│  Input: one new token per step                            │
│  Output: one new token per step                           │
│  Compute: sequential — attends over growing KV cache      │
│  Duration: ~proportional to n_new_tokens                  │
└─────────────────────────────────────────────────────────┘
```

**Why this matters for system design:**
- Prefill is **compute-bound** (high arithmetic intensity) — benefits from large batches
- Decode is **memory-bound** (low arithmetic intensity) — bottlenecked by KV cache bandwidth
- Production systems handle these phases differently (e.g. chunked prefill, continuous batching)


In [ ]:
# ── Profile prefill vs decode timing ─────────────────────────────────
def profile_prefill_decode(model, tokenizer, prompt_lengths, max_new=20, device="cpu"):
    results = []
    model.eval()

    for plen in prompt_lengths:
        # Construct a prompt of exactly plen tokens
        dummy_ids = torch.randint(100, 50000, (1, plen)).to(device)

        # Prefill
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model(dummy_ids, use_cache=True)
        prefill_time = time.perf_counter() - t0
        past = out.past_key_values

        # Decode: average over max_new steps
        decode_times = []
        current = dummy_ids[:, -1:]
        with torch.no_grad():
            for _ in range(max_new):
                t0 = time.perf_counter()
                out = model(current, past_key_values=past, use_cache=True)
                decode_times.append(time.perf_counter() - t0)
                past    = out.past_key_values
                current = out.logits[:, -1:, :].argmax(dim=-1)

        results.append({
            "prompt_len":   plen,
            "prefill_ms":   prefill_time * 1000,
            "decode_ms":    np.mean(decode_times) * 1000,
        })
        print(f"  prompt={plen:4d} tokens | prefill={prefill_time*1000:6.1f}ms | "
              f"decode={np.mean(decode_times)*1000:5.1f}ms/tok")

    return results


print("Profiling prefill vs decode (GPT-2 on CPU — may take ~30s):")
results = profile_prefill_decode(model, tokenizer,
                                  prompt_lengths=[16, 64, 128, 256],
                                  max_new=10, device=device)


In [ ]:
# ── Plot prefill vs decode scaling ───────────────────────────────────
prompt_lens  = [r["prompt_len"]  for r in results]
prefill_ms   = [r["prefill_ms"]  for r in results]
decode_ms    = [r["decode_ms"]   for r in results]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(prompt_lens, prefill_ms, "o-", color="#3498db", lw=2, markersize=8)
axes[0].set_xlabel("Prompt length (tokens)")
axes[0].set_ylabel("Prefill time (ms)")
axes[0].set_title("Prefill time vs prompt length")
axes[0].grid(alpha=0.3)

axes[1].plot(prompt_lens, decode_ms, "s-", color="#e67e22", lw=2, markersize=8)
axes[1].set_xlabel("Prompt length (tokens)")
axes[1].set_ylabel("Decode time per token (ms)")
axes[1].set_title("Decode latency vs prompt length
(grows with KV cache size)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 7. Stop Conditions and Token Budgets

In [ ]:
# ── Demonstrate stop conditions ───────────────────────────────────────
def generate_with_stops(
    model, tokenizer, prompt: str,
    max_new_tokens: int = 100,
    stop_strings: list[str] = None,
    device: str = "cpu",
) -> dict:
    """Generation with configurable stop conditions."""
    model.eval()
    stop_strings = stop_strings or []
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    generated = input_ids.clone()
    stop_reason = "max_length"

    with torch.no_grad():
        for step in range(max_new_tokens):
            out        = model(generated, use_cache=False)
            next_tok   = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated  = torch.cat([generated, next_tok], dim=1)

            # Stop 1: EOS token
            if next_tok.item() == tokenizer.eos_token_id:
                stop_reason = "eos_token"
                break

            # Stop 2: Stop string match
            decoded_so_far = tokenizer.decode(generated[0], skip_special_tokens=True)
            for s in stop_strings:
                if decoded_so_far.endswith(s):
                    stop_reason = f"stop_string: {s!r}"
                    break
            else:
                continue
            break

    text    = tokenizer.decode(generated[0], skip_special_tokens=True)
    n_new   = generated.shape[1] - input_ids.shape[1]
    return {"text": text, "n_new": n_new, "stop_reason": stop_reason}


result = generate_with_stops(
    model, tokenizer,
    prompt="The quick brown fox",
    max_new_tokens=40,
    stop_strings=[".", "\n"],
    device=device,
)
print(f"Output      : {result['text']}")
print(f"New tokens  : {result['n_new']}")
print(f"Stop reason : {result['stop_reason']}")


## 8. Engineering Insights

### 8.1 Key Latency Metrics

In production LLM systems, two latency metrics matter most:

| Metric | Definition | Optimise by |
|---|---|---|
| **TTFT** (Time to First Token) | Time from request to first output token | Fast prefill, small model, speculative decoding |
| **TPOT** (Time per Output Token) | Average time between successive output tokens | KV cache efficiency, quantisation, batching |
| **E2E latency** | Total time = TTFT + TPOT × n_tokens | Both |

**Throughput** (for batch serving):
$$
\text{throughput} = \frac{\text{batch size} \times n_{\text{tokens}}}{\text{wall clock time}} \quad [\text{tokens/sec}]
$$


In [ ]:
# ── Measure TTFT and TPOT ────────────────────────────────────────────
def measure_latency(model, tokenizer, prompt: str,
                    max_new_tokens: int = 20, device: str = "cpu"):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    past      = None
    current   = input_ids
    times     = []

    with torch.no_grad():
        for step in range(max_new_tokens):
            t0  = time.perf_counter()
            out = model(current, past_key_values=past, use_cache=True)
            elapsed = time.perf_counter() - t0

            times.append(elapsed)
            past    = out.past_key_values
            current = out.logits[:, -1:, :].argmax(dim=-1)

            if current.item() == tokenizer.eos_token_id:
                break

    ttft = times[0] * 1000       # first token = prefill + first decode
    tpot = np.mean(times[1:]) * 1000 if len(times) > 1 else 0
    e2e  = sum(times) * 1000

    print(f"Prompt length : {input_ids.shape[1]} tokens")
    print(f"Output tokens : {len(times)}")
    print(f"TTFT          : {ttft:.1f} ms")
    print(f"TPOT          : {tpot:.1f} ms/token")
    print(f"E2E latency   : {e2e:.1f} ms")
    print(f"Throughput    : {len(times) / (e2e/1000):.1f} tokens/sec")
    return times


times = measure_latency(
    model, tokenizer,
    prompt="Explain the difference between supervised and unsupervised learning in machine learning.",
    max_new_tokens=25,
    device=device,
)


In [ ]:
# ── Visualise TTFT vs TPOT ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

steps = range(1, len(times) + 1)
colors = ["#e74c3c"] + ["#3498db"] * (len(times) - 1)
bars = ax.bar(steps, [t * 1000 for t in times], color=colors, edgecolor="white", linewidth=0.5)

ax.set_xlabel("Generation step")
ax.set_ylabel("Time (ms)")
ax.set_title("Per-step latency: TTFT (red) vs TPOT (blue)")
ax.grid(axis="y", alpha=0.3)

# Legend
ax.bar(0, 0, color="#e74c3c", label=f"TTFT = {times[0]*1000:.1f}ms")
ax.bar(0, 0, color="#3498db", label=f"TPOT avg = {np.mean(times[1:])*1000:.1f}ms")
ax.legend()

plt.tight_layout()
plt.show()


### 8.2 KV Cache Memory Planning

At deployment, you must decide **before serving** how much GPU memory to reserve for the KV cache.

Key variables:
- **Batch size** $B$: number of concurrent requests
- **Max sequence length** $T$: prompt + max output tokens
- **Model config**: $n_\text{layers}$, $n_\text{kv\_heads}$, $d_k$

$$
\text{KV cache total} = B \times 2 \times n_\text{layers} \times n_\text{kv\_heads} \times d_k \times T \times \text{bytes}
$$

This competes directly with model weights for GPU memory:

```
GPU memory budget
├── Model weights   (fixed)
├── Activations     (proportional to batch × seq)
└── KV cache        (proportional to batch × max_seq × layers)
```


In [ ]:
# ── KV cache memory planner ──────────────────────────────────────────
def kv_cache_memory_gb(
    batch_size: int,
    max_seq_len: int,
    n_layers: int,
    n_kv_heads: int,
    d_k: int,
    dtype_bytes: int = 2,   # float16
) -> float:
    total = 2 * batch_size * n_layers * n_kv_heads * d_k * max_seq_len * dtype_bytes
    return total / 1024**3


# Model configs
models_cfg = {
    "GPT-2 (117M)":     dict(n_layers=12,  n_kv_heads=12, d_k=64),
    "LLaMA-3-8B":       dict(n_layers=32,  n_kv_heads=8,  d_k=128),
    "LLaMA-3-70B":      dict(n_layers=80,  n_kv_heads=8,  d_k=128),
    "Mistral-7B":       dict(n_layers=32,  n_kv_heads=8,  d_k=128),
}

batch_sizes  = [1, 4, 8, 16, 32]
max_seq      = 4096

print(f"KV Cache Memory (GB) — max_seq={max_seq}, float16")
print(f"{'Model':<22}", end="")
for b in batch_sizes: print(f"  bs={b:>2}", end="")
print()
print("-" * 65)

for name, cfg in models_cfg.items():
    print(f"{name:<22}", end="")
    for b in batch_sizes:
        mem = kv_cache_memory_gb(b, max_seq, **cfg)
        flag = " ⚠️" if mem > 40 else ""
        print(f"  {mem:>4.1f}G", end="")
    print()


### 8.3 Throughput vs Latency Tradeoff

**Batching** is the primary lever for throughput:
- Larger batch → more tokens processed per GPU second → higher throughput
- Larger batch → longer wait for first token → higher latency

```
Single request:   low latency,   low throughput
Large batch:      high latency,  high throughput
```

Production systems use **continuous batching** (iteration-level scheduling) to balance both:
new requests are added to the batch mid-generation rather than waiting for the whole batch to finish.


In [ ]:
# ── Simulate batch throughput ─────────────────────────────────────────
def batch_generate(model, tokenizer, prompts: list[str],
                   max_new_tokens: int = 20, device: str = "cpu"):
    """Simple batched generation (static batching)."""
    tokenizer.pad_token = tokenizer.eos_token
    inputs = tokenizer(prompts, return_tensors="pt",
                       padding=True, truncation=True).to(device)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.perf_counter() - t0
    total_tokens = out.shape[0] * max_new_tokens
    return elapsed, total_tokens


prompts_pool = [
    "The history of artificial intelligence",
    "Python is a popular programming language because",
    "The largest ocean in the world is",
    "Climate change affects the planet by",
]

print(f"{'Batch size':>12} {'Time (s)':>10} {'Total tokens':>14} {'Throughput (tok/s)':>20}")
print("-" * 62)
for bs in [1, 2, 4]:
    prompts_batch = (prompts_pool * 10)[:bs]
    elapsed, n_tok = batch_generate(model, tokenizer, prompts_batch,
                                     max_new_tokens=15, device=device)
    throughput = n_tok / elapsed
    print(f"{bs:>12} {elapsed:>10.2f} {n_tok:>14} {throughput:>20.1f}")


## 9. Token Streaming

In [ ]:
# ── Streaming generation (print each token as it arrives) ────────────
from transformers import TextStreamer

print("Streaming output (tokens print as they are generated):")
print("-" * 50)

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
input_ids = tokenizer.encode(
    "The future of artificial intelligence is",
    return_tensors="pt"
).to(device)

with torch.no_grad():
    model.generate(
        input_ids,
        max_new_tokens=40,
        do_sample=True,
        temperature=0.8,
        streamer=streamer,
    )
print()
print("-" * 50)
print("Each token above was emitted as soon as it was sampled.")


## 10. Exercises

1. **KV cache verification:** After running `generate_with_cache`, inspect `past_key_values`. What is the shape of the key tensor for layer 0? Verify it grows by 1 along the sequence dimension at each step.

2. **Temperature sweep:** Generate 5 continuations of the same prompt at temperatures $\tau \in \{0.1, 0.5, 1.0, 1.5, 2.0\}$. How does the output change? Compute the token-level entropy of the logit distribution at each temperature.

3. **Prefill scaling:** Measure prefill time for prompt lengths $[64, 128, 256, 512, 1024]$. Fit a regression line. Is it linear or quadratic? Why?

4. **Batch throughput curve:** Extend the batch throughput experiment to batch sizes $[1, 2, 4, 8, 16]$. Plot throughput vs batch size. At what batch size does throughput plateau? What is the bottleneck?

5. **Stop string robustness:** Implement a stop-string detector that handles partial matches at the boundary of the generated buffer (e.g. if the stop string is `"\n\n"` and the last two tokens are `"\n"` and `"\n"`, it should stop even though neither token alone matches).

---

## 11. Key Takeaways

| Concept | Summary |
|---|---|
| **Autoregressive generation** | One token per step; each step conditions on all prior tokens |
| **Prefill** | Parallel processing of the full prompt — compute-bound |
| **Decode** | Sequential token-by-token generation — memory-bound |
| **KV cache** | Caches K and V for past tokens; reduces decode from $\mathcal{O}(T^2)$ to $\mathcal{O}(T)$ |
| **TTFT** | Time to first token; dominated by prefill speed |
| **TPOT** | Time per output token; dominated by KV cache bandwidth |
| **Batching** | Key lever for throughput; trades latency for efficiency |
| **Streaming** | Emit tokens as generated — reduces perceived latency |
